# 09. Context i State: dwa r??ne ?r?d?a danych

W lekcjach 06 i 07 poznali?my `state` i `context` osobno. Teraz prze?ledzimy **ten sam przebieg** i sprawdzimy, sk?d ka?da warto?? pochodzi, kto mo?e j? odczyta? i co dzieje si? po nast?pnym wywo?aniu. Przyk?ad u?ywa jawnie zaprogramowanych odpowiedzi zamiast prawdziwego LLM, wi?c wynik jest powtarzalny. Nie symulujemy logowania: kontekst traktujemy jak dane, kt?re aplikacja wcze?niej sprawdzi?a.

[Dokumentacja LangChain: ?r?d?a danych i context engineering](https://docs.langchain.com/oss/python/langchain/context-engineering) ? [LangGraph: state i runtime context](https://docs.langchain.com/oss/python/langgraph/graph-api)

In [ ]:
# Jedyna kom?rka z importami.
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import NotRequired, TypedDict

from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.faults import ScriptedModel


## 1. Najpierw zwyk?y Python

`state` to dane, kt?re **zmieniaj? si? podczas pracy**: historia wiadomo?ci, rezultat narz?dzia, licznik, status zadania. `context` to dane, kt?re **aplikacja dostarcza do konkretnego wywo?ania**: ustalony identyfikator u?ytkownika, j?zyk, uprawnienia lub uchwyt do us?ugi. Samo przekazanie kontekstu nie dopisuje go do promptu ani stanu.

Poni?sze s?owniki s? tylko obrazkiem. W?a?ciwe API pojawi si? dalej.

In [ ]:
state = {"messages": ["Jaki mam plan?"], "turns": 0}
context = {"viewer_id": "u-101", "language": "pl"}
state["turns"] += 1
state, context


| Pytanie | State | Runtime context |
|---|---|---|
| Kto podaje warto??? | wej?cie u?ytkownika, model, narz?dzie lub kod grafu | kod aplikacji przy `invoke(..., context=...)` |
| Czy mo?e zmienia? si? podczas przebiegu? | tak, przez zwracane aktualizacje | traktuj jako wej?cie tylko do odczytu |
| Czy checkpointer mo?e zachowa? j? dla `thread_id`? | tak | nie; dostarcz ponownie przy kolejnym wywo?aniu |
| Czy model widzi j? automatycznie? | tylko wiadomo?ci i pola jawnie w??czone do ??dania | nie; dopiero gdy kod umie?ci warto?? w promptcie lub wyniku narz?dzia |
| Jak j? odczyta?? | `result["messages"]`, `runtime.state`, argument `state` hooka/w?z?a | `runtime.context` |

**Granica zaufania:** `viewer_id` i uprawnienia bierze serwer z wcze?niej sprawdzonej sesji. Tekst u?ytkownika ani `state` wygenerowany przez model nie powinny nadawa? uprawnie?. Checkpointer przechowuje stan w?tku, a nie autoryzuje ??da?.

## 2. Kontekst podaje aplikacja, narz?dzie odczytuje stan

`RequestContext` jest ma?ym, niemutowalnym kontraktem. S?ownik profili zast?puje baz?. Model zna tylko nazw? narz?dzia; `viewer_id` nie jest argumentem narz?dzia, kt?ry m?g?by wymy?li? model. `runtime.state` w narz?dziu pokazuje aktualny stan agenta, a `runtime.context` dane bie??cego wywo?ania.

In [ ]:
@dataclass(frozen=True)
class RequestContext:
    viewer_id: str
    language: str


PROFILES = {"u-101": "Plus", "u-202": "Basic"}
observed = []


@tool
def my_plan(runtime: ToolRuntime[RequestContext]) -> str:
    """Odczytaj plan u?ytkownika ustalonego przez aplikacj?."""
    # ID pochodzi z context. Model nie przekazuje go jako argumentu narz?dzia.
    viewer_id = runtime.context.viewer_id
    # State zawiera wiadomo?ci tej rozmowy; nie jest ?r?d?em to?samo?ci.
    observed.append({"viewer_id": viewer_id, "messages_seen": len(runtime.state["messages"])})
    return PROFILES[viewer_id]


## 3. Agent LangChain: wej?cie, narz?dzie, wynik

Sztuczny model wykona zaplanowane wywo?anie narz?dzia i zwr?ci zaplanowan? odpowied?. Nie ocenia sam, kt?ry plan jest poprawny. Poprawn? warto?? zwraca Python z `PROFILES`; w prawdziwej aplikacji by?by to odczyt z uprawnionej bazy. U?ywamy `context_schema` jako kontraktu dla `context=...`.

In [ ]:
scripted_model = ScriptedModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "my_plan", "args": {}, "id": "call-a"}]),
    AIMessage(content="Plan odczytany z narz?dzia."),
    AIMessage(content="", tool_calls=[{"name": "my_plan", "args": {}, "id": "call-b"}]),
    AIMessage(content="Drugi plan odczytany z narz?dzia."),
])
agent = create_agent(
    model=scripted_model,
    tools=[my_plan],
    context_schema=RequestContext,
    checkpointer=InMemorySaver(),
)
thread = {"configurable": {"thread_id": "lekcja-09"}}
first = agent.invoke(
    {"messages": [("user", "Jaki mam plan?")]},
    context=RequestContext(viewer_id="u-101", language="pl"),
    config=thread,
)
[(m.type, m.content) for m in first["messages"]], observed


`first["messages"]` jest stanem po przebiegu. Zawiera wiadomo?? u?ytkownika, pro?b? modelu o narz?dzie, wynik narz?dzia i odpowied? modelu. W odpowiedzi sztucznego modelu nie ma jeszcze nazwy planu: tekst jest zaprogramowany. To celowo oddziela **decyzj? modelu** od **danych zwr?conych przez Python**.

In [ ]:
second = agent.invoke(
    {"messages": [("user", "A teraz m?j plan?")]},
    context=RequestContext(viewer_id="u-202", language="en"),
    config=thread,
)
[(m.type, m.content) for m in second["messages"][-4:]], observed[-1]


Ten sam `thread_id` zachowa? histori? wiadomo?ci, lecz **nie zamrozi? kontekstu**: druga tura dosta?a `u-202`. W aplikacji produkcyjnej nie zmieniaj w ten spos?b w?a?ciciela cudzego w?tku; tutaj to eksperyment pokazuj?cy granic? API. W?a?ciciela w?tku trzeba sprawdzi? poza grafem. `InMemorySaver` trzyma dane tylko przez ?ycie tego obiektu w bie??cym procesie.

In [ ]:
saved = agent.get_state(thread)
len(saved.values["messages"]), [m.content for m in saved.values["messages"] if m.type == "tool"]


## 4. Delikatne spojrzenie pod spodem: LangGraph

Agent LangChain dzia?a na grafie. Graf ma **w?z?y** wykonuj?ce kod i **state**, kt?re przechodzi mi?dzy nimi. `StateGraph` opisuje kszta?t tego stanu. W?ze? dostaje bie??cy stan oraz `Runtime` z kontekstem. Zwraca **cz??ciow? aktualizacj?**, a graf sk?ada nowy stan. Tu jest tylko jeden w?ze?; nie uczymy jeszcze trasowania ani reducer?w.

In [ ]:
class VisitState(TypedDict):
    question: str
    turns: NotRequired[int]
    answer: NotRequired[str]


def answer_question(state: VisitState, runtime: Runtime[RequestContext]) -> dict:
    # State mo?e zmienia? si? mi?dzy turami; context jest podany dla tej tury.
    turns = state.get("turns", 0) + 1
    plan = PROFILES[runtime.context.viewer_id]
    return {"turns": turns, "answer": f"Tura {turns}: plan {plan}. Pytanie: {state['question']}"}


builder = StateGraph(VisitState, context_schema=RequestContext)
builder.add_node("answer", answer_question)
builder.add_edge(START, "answer")
builder.add_edge("answer", END)
graph = builder.compile(checkpointer=InMemorySaver())
graph_thread = {"configurable": {"thread_id": "graf-09"}}


In [ ]:
graph_first = graph.invoke(
    {"question": "Jaki plan?"},
    context=RequestContext("u-101", "pl"),
    config=graph_thread,
)
graph_second = graph.invoke(
    {"question": "A teraz?"},
    context=RequestContext("u-202", "en"),
    config=graph_thread,
)
graph_first, graph_second


`turns` ro?nie, bo graf odczytuje poprzedni zapisany state. `answer` i `question` s? polami stanu. `viewer_id` s?u?y w?z?owi do obliczenia odpowiedzi, ale nie pojawia si? jako pole stanu. Bez checkpointera ka?de wywo?anie zacz??oby od wej?ciowego stanu. Nowy `thread_id` te? zacznie od pocz?tku.

In [ ]:
other_thread = graph.invoke(
    {"question": "Jaki plan?"},
    context=RequestContext("u-101", "pl"),
    config={"configurable": {"thread_id": "graf-inny"}},
)
other_thread


## 5. Jak wybiera? poprawne miejsce na dane?

- **Wiadomo?ci, post?p zadania, wynik narz?dzia:** state. Aktualizuj przez zwrot z w?z?a/hooka albo `Command(update=...)` z narz?dzia, nie przez przypadkow? mutacj? `runtime.state`.
- **To?samo??, zakres uprawnie?, j?zyk bie??cego ??dania, klient bazy:** runtime context. Ustalaj w aplikacji i podawaj przy ka?dym `invoke`. Nie wk?adaj sekret?w do wiadomo?ci.
- **Instrukcje i fakty potrzebne modelowi teraz:** jawnie do promptu lub wyniku narz?dzia. Samo `context=...` niczego modelowi nie pokazuje.
- **Wiedza z innych rozm?w:** osobny trwa?y magazyn lub baza. Checkpointer stanu jednej rozmowy nie jest takim magazynem.

**?wiczenie:** dodaj do `RequestContext` pole `region`, a do `VisitState` pole `last_plan`. W w??le odczytaj pierwsze z `runtime.context`, drugie aktualizuj przez zwr?cony s?ownik. Sprawd? dwa w?tki i dwa r??ne konteksty dla tego samego w?tku.